[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/filippolonghi/medleydb-mert-instrument-classification/blob/main/notebooks/03_mert_finetuning_experiments.ipynb)


# 03 - Direct MERT and Fine-Tuning Ablation

## Research question

Does direct audio training or partial MERT fine-tuning improve over cached frozen embeddings enough to justify the compute cost?

## Approach

These configs use the same largest-balanced MedleyDB instrument subset but load audio on the fly. Batch size 1, mixed precision, gradient checkpointing, and gradient accumulation are set for Colab-style GPUs.

## What is fixed

Unless a selected config says otherwise: MedleyDB instrument labels, the `largest_balanced` split, MERT-v1-95M, 5 s segments, validation-based model selection, and test metrics saved only after training.

## What is varied

Frozen direct MERT, fine-tuning the last 1 transformer layer, and fine-tuning the last 2 transformer layers.

## Expected interpretation

Fine-tuning should be treated as a compute-heavy confirmation experiment; compare against the best cached frozen representation from notebooks 01 and 02.


## What you can change

- `PROJECT_ROOT`, `RUN_ROOT`, and `MEDLEYDB_ROOT` for local vs Colab/Drive paths.
- `EXPERIMENT_GROUPS`, `DATASET_GROUPS`, `MIXTURE_GROUPS`, and `SELECTED_EXPERIMENTS` to run one config at a time.
- `REPLACE_EXISTING = True` only when intentionally overwriting a completed run.


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys
import yaml
import pandas as pd

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/content/medleydb-mert-instrument-classification"))
RUN_ROOT = Path(os.environ.get("RUN_ROOT", "/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1"))
MEDLEYDB_ROOT = Path(os.environ.get("MEDLEYDB_ROOT", "/content/drive/MyDrive/medleydb_mert_project/MedleyDB"))

os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["RUN_ROOT"] = str(RUN_ROOT)
os.environ["MEDLEYDB_ROOT"] = str(MEDLEYDB_ROOT)

SUBSET_PROFILE = "largest_balanced"
LABEL_GRANULARITY = "medleydb_instrument"
RESULTS_DIR = RUN_ROOT / "results"
CHECKPOINTS_DIR = RUN_ROOT / "checkpoints"
METADATA_DIR = RUN_ROOT / "data" / "metadata"
CACHE_ROOT = RUN_ROOT / "data" / "cache"
REPORTS_DIR = RUN_ROOT / "data" / "reports"
for path in [RESULTS_DIR, CHECKPOINTS_DIR, METADATA_DIR, CACHE_ROOT, REPORTS_DIR, RUN_ROOT / "configs"]:
    path.mkdir(parents=True, exist_ok=True)

if PROJECT_ROOT.exists():
    os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Run root:", RUN_ROOT)
print("MedleyDB root:", MEDLEYDB_ROOT)
print("Subset profile:", SUBSET_PROFILE)
print("Label granularity:", LABEL_GRANULARITY)


## Compute Warning

Full fine-tuning is intentionally disabled in the final configs. Try full fine-tuning only on a small subset and only after explicitly enabling it in a separate config.

In [ ]:
def q(path):
    return shlex.quote(str(path))


def run(cmd, check=True):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, check=check)


def load_config(config_path):
    return yaml.safe_load(Path(config_path).read_text(encoding="utf-8"))


def run_experiment_config(config_path, *, extract_embeddings=True, replace_existing=False):
    config_path = Path(config_path)
    cfg = load_config(config_path)
    approach = cfg.get("approach", "frozen_embeddings")
    if approach == "frozen_embeddings" and extract_embeddings:
        run(f"python -m src.features.extract_mert_embeddings --experiment-config {q(config_path)} --batch-size 1 --device auto")
    elif approach == "polyphonic_multilabel" and extract_embeddings:
        run(f"python -m src.features.extract_mert_mixture_embeddings --experiment-config {q(config_path)} --batch-size 1 --device auto")
    args = f"--config {q(config_path)}"
    if replace_existing:
        args += " --replace-existing"
    return run(f"python -m src.experiments.run_experiment {args}")


def run_selected(configs, *, extract_embeddings=True, replace_existing=False):
    for config in configs:
        run_experiment_config(config, extract_embeddings=extract_embeddings, replace_existing=replace_existing)


In [ ]:
EXPERIMENT_GROUPS = {
    "direct_and_partial_finetune": [
        "configs/experiments/direct_largest_balanced_medleydb_mert95_frozen.yaml",
        "configs/experiments/finetune_largest_balanced_medleydb_mert95_last1.yaml",
        "configs/experiments/finetune_largest_balanced_medleydb_mert95_last2.yaml",
    ],
}
SELECTED_EXPERIMENTS = [
    "configs/experiments/direct_largest_balanced_medleydb_mert95_frozen.yaml",
]
REPLACE_EXISTING = False

run_selected(SELECTED_EXPERIMENTS, extract_embeddings=False, replace_existing=REPLACE_EXISTING)
